# NB01 — Extract MIMIC-IV

**Purpose.** Run the harmonised MIMIC-IV extraction queries (validated in NB00) and store the raw aggregates as parquet files on Drive. No modelling happens here.



| | |
|---|---|
| **Inputs** | NB00 dry-run all OK |
| **Outputs** | `data/raw/mimiciv_*.parquet` |
| **Free-Colab runtime** | 15–35 min (BigQuery does the heavy lifting) |

**How to run:** `Runtime → Run all`. If Colab disconnects, just run all again — finished steps are cached.
At the end, download the zip printed in the last cell into your local `Plan A/v4/results/` folder.

In [ ]:
# ==== Setup (identical in every notebook) ====
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ.setdefault('TRANSPORT_BASE', '/content/drive/MyDrive/Plan A/v4')
BASE = os.environ['TRANSPORT_BASE']
sys.path.insert(0, os.path.join(BASE, 'code'))
import importlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import transport_utils as tu
importlib.reload(tu)
warnings.filterwarnings('ignore')
tu.set_style()
OUT = tu.init_notebook('NB01')

In [ ]:
# ==== BigQuery connection ====
from google.colab import auth
from google.cloud import bigquery
auth.authenticate_user()
PROJECT = 'mimic-rs'          # <-- your Google Cloud project with PhysioNet access
client = bigquery.Client(project=PROJECT)
import sql_queries as sq
importlib.reload(sq)
print('Connected to BigQuery project:', PROJECT)

In [ ]:
FORCE = False    # set True to re-download even if the file already exists
NAMES = ['mimic_cohort', 'mimic_vitals', 'mimic_labs', 'mimic_gcs']
shapes = {}
for name in NAMES:
    path = os.path.join(tu.DATA_RAW, name + '.parquet')
    if os.path.exists(path) and not FORCE:
        df = pd.read_parquet(path)
        print(f'[skip] {name} already on Drive: {df.shape}')
    else:
        print(f'[run ] {name} ...', flush=True)
        df = sq.run_to_parquet(client, name, path)
        print(f'       saved {df.shape} -> {path}')
    shapes[name] = list(df.shape)

In [ ]:
# Quick sanity checks (aggregate only -- never print patient-level rows)
prefix = 'mimic'
coh = pd.read_parquet(os.path.join(tu.DATA_RAW, prefix + '_cohort.parquet'))
print('cohort rows:', len(coh), '| first stays (rn==1):', int((pd.to_numeric(coh['rn']) == 1).sum()))
vit = pd.read_parquet(os.path.join(tu.DATA_RAW, prefix + '_vitals.parquet'))
lab = pd.read_parquet(os.path.join(tu.DATA_RAW, prefix + '_labs.parquet'))
cov = []
for name, df in [('vitals', vit), ('labs', lab)]:
    g = df.groupby('var').agg(stays=('stay_id', 'nunique'), median_of_mean=('vmean', 'median'),
                              p01=('vmin', lambda s: s.quantile(0.01)), p99=('vmax', lambda s: s.quantile(0.99)),
                              median_count=('n_raw', 'median')).reset_index()
    g.insert(0, 'table', name)
    cov.append(g)
cov = pd.concat(cov)
cov['coverage_%_first_stays'] = 100 * cov['stays'] / int((pd.to_numeric(coh['rn']) == 1).sum())
tu.save_table(cov.round(3), 'variable_coverage')

In [ ]:
tu.finish_notebook({'shapes': shapes, 'variable_coverage': cov.round(3).to_dict('records')})